# Notebook 9 – Random Forest

## Topics
- Ensemble Learning
- Bagging
- Random Forest
- Multiple Decision Trees
- Random Feature Selection
- Voting
- Feature Importance
- Overfitting
- Advantages
- Limitations
- Decision Tree vs Random Forest

## 1. Ensemble Learning

**Ensemble Learning** combines multiple machine learning models to produce a stronger overall model.

The idea is that several models working together can often make better predictions than a single model.

## 2. Bagging

**Bagging (Bootstrap Aggregating)** trains multiple models on different bootstrap samples of the training data.

The predictions of these models are then combined.

For classification, the final prediction is commonly based on **majority voting**.

## 3. Random Forest

**Random Forest** is an ensemble algorithm made up of many Decision Trees.

It combines two types of randomness:
1. Random sampling of training data using bootstrap samples.
2. Random selection of features when splitting nodes.

The final prediction is obtained by combining the predictions of all trees.

## 4. Multiple Decision Trees

A Random Forest does not depend on one Decision Tree. It creates many different trees.

Because the trees see different samples and feature subsets, they make somewhat different predictions. Combining them makes the overall model more stable.

## 5. Random Feature Selection

At each split, Random Forest considers only a random subset of features instead of always considering every feature.

This makes the trees less similar to each other and helps reduce correlation between trees.

## 6. Voting

For classification, every tree predicts a class.

The Random Forest combines these predictions using **majority voting**.

Example:
- Tree 1 → Class A
- Tree 2 → Class A
- Tree 3 → Class B
- Tree 4 → Class A

Final prediction → **Class A**

## 7. Feature Importance

**Feature importance** tells us how useful each feature was for making predictions.

In scikit-learn, Random Forest provides the `feature_importances_` attribute.

## 8. Overfitting

A single Decision Tree can easily become very deep and overfit the training data.

Random Forest generally reduces this problem because it averages many different trees.

However, Random Forest can still overfit, especially with noisy data or poorly chosen hyperparameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 9. Load the Dataset

We will use the Iris dataset for classification.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('Feature names:', iris.feature_names)
print('Classes:', iris.target_names)

## 10. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 11. Train a Single Decision Tree

We first train one Decision Tree so that we can compare it with Random Forest.

In [ ]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)

dt_predictions = decision_tree.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_predictions)

print('Decision Tree Accuracy:', dt_accuracy)

## 12. Train a Random Forest

`n_estimators` specifies the number of decision trees in the forest.

`random_state` makes the result reproducible.

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest.fit(X_train, y_train)

rf_predictions = random_forest.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)

print('Random Forest Accuracy:', rf_accuracy)

## 13. Random Forest Classification Report

In [ ]:
print('Random Forest Classification Report')
print(classification_report(y_test, rf_predictions, target_names=iris.target_names))

## 14. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, rf_predictions)
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
).plot()
plt.title('Random Forest Confusion Matrix')
plt.show()

## 15. Feature Importance

We can inspect which Iris features were most important to the Random Forest.

In [ ]:
importances = random_forest.feature_importances_
indices = np.argsort(importances)[::-1]

print('Feature Importance:')
for index in indices:
    print(f'{iris.feature_names[index]}: {importances[index]:.4f}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [iris.feature_names[i] for i in indices], rotation=30, ha='right')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

## 16. Visualize One Tree from the Random Forest

A Random Forest contains many trees. We can visualize one of them to understand its structure.

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    random_forest.estimators_[0],
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    max_depth=3,
    rounded=True
)
plt.title('One Decision Tree from the Random Forest')
plt.show()

## 17. Effect of Number of Trees

We can test different values of `n_estimators`.

In [ ]:
tree_counts = [1, 5, 10, 50, 100, 200]
tree_accuracies = []

for n in tree_counts:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    tree_accuracies.append(accuracy_score(y_test, predictions))

for n, acc in zip(tree_counts, tree_accuracies):
    print(f'n_estimators={n}: Accuracy={acc:.3f}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(tree_counts, tree_accuracies, marker='o')
plt.xlabel('Number of Trees')
plt.ylabel('Test Accuracy')
plt.title('Effect of Number of Trees')
plt.grid(True)
plt.show()

## 18. Decision Tree vs Random Forest

In [ ]:
print(f'Decision Tree Accuracy: {dt_accuracy:.3f}')
print(f'Random Forest Accuracy: {rf_accuracy:.3f}')

| Feature | Decision Tree | Random Forest |
|---|---|---|
| Number of models | One tree | Many trees |
| Training | Single tree | Multiple trees |
| Feature selection | Usually considers available features at a split | Random subset of features |
| Prediction | One tree's prediction | Combined prediction of many trees |
| Overfitting | More prone | Generally less prone |
| Interpretability | Easier | More difficult |
| Accuracy | Can be good | Often more robust |
| Computation | Lower | Higher |

## 19. Advantages of Random Forest

- Usually more robust than a single Decision Tree.
- Reduces variance through averaging multiple trees.
- Can handle non-linear relationships.
- Provides feature importance.
- Usually requires less preprocessing than distance-based algorithms.
- Can work well for both classification and regression.

## 20. Limitations of Random Forest

- More computationally expensive than one Decision Tree.
- Can require more memory because many trees are stored.
- Less interpretable than a single Decision Tree.
- Predictions can be slower when the forest contains many trees.
- It can still overfit in some situations.

# 21. Viva Summary

1. **Ensemble Learning:** Combines multiple models to improve overall performance.
2. **Bagging:** Trains models on different bootstrap samples and combines their predictions.
3. **Random Forest:** An ensemble of multiple Decision Trees.
4. **Multiple trees:** Different trees are created using random samples and feature subsets.
5. **Random feature selection:** A random subset of features is considered at each split.
6. **Voting:** For classification, tree predictions are combined using majority voting.
7. **Feature importance:** Measures how useful each feature is for the forest's predictions.
8. **Overfitting:** Random Forest generally reduces the overfitting risk of a single tree, but it is not completely immune.
9. **n_estimators:** Number of trees in the Random Forest.
10. **random_state:** Makes results reproducible.
11. **Decision Tree vs Random Forest:** A Decision Tree is one model, while Random Forest combines many trees.
12. **Main advantage:** Better stability and generalization than many individual trees.
13. **Main limitation:** Higher computational cost and lower interpretability than a single tree.